# Projeto Filtro Rejeita-Faixa FIR — Pipeline de Validação

**Disciplina:** DSP II  
**Objetivo:** Projetar, gerar os coeficientes e validar em Python um filtro FIR
rejeita-faixa para supressão do ruído tonal em **874,98 Hz**, e embarcar
o filtro no kit STM32F4 Discovery + Wolfson Pi Audio.

### Fluxo

```
 ┌─────────┐      ┌──────────────┐      ┌──────────────┐      ┌───────┐
 │  pyfda  │ ───▶ │ coeffs_FIR   │ ───▶ │ Le_CSV_...py │ ───▶ │ .h    │
 │ /firwin │      │    .csv      │      │              │      │(CMSIS)│
 └─────────┘      └──────────────┘      └──────────────┘      └───┬───┘
                                                                   ▼
                                                          ┌────────────────┐
                                                          │  STM32F4 +     │
                                                          │  Wolfson Pi    │
                                                          │  (main.c)      │
                                                          └────────────────┘
                                                                   ▲
            ┌──────────────────────┐          ┌──────────────┐     │
            │  audio-teste-ruido-  │ ─(PC)─▶  │  LINE-IN     │ ────┘
            │        G1.wav       │          │  do codec    │
            └──────────────────────┘          └──────────────┘
            ┌──────────────────────┐          (saída: fone/linha out)
            │ ruído branco 200 Hz  │ ─(PC)─▶
            │      – 20 kHz.wav    │
            └──────────────────────┘
```

Este notebook simula em Python **exatamente** o que o kit faz em tempo real:
- Reamostra os sinais para **Fs = 48 kHz** (taxa do codec Wolfson)
- Aplica o filtro FIR projetado com N=199, janela Hamming, bandstop 400–1350 Hz
- Salva os áudios filtrados e produz gráficos FFT / espectrograma

## 0. Imports e parâmetros globais

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
import soundfile as sf
import librosa
import librosa.display
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# Especificações validadas do projeto (conforme notebook de análise)
# =====================================================================
FS_KIT   = 48_000        # Taxa de amostragem do kit Wolfson/STM32
NUMTAPS  = 199           # Ordem + 1 (limite <200 imposto pelo kit)
F_CORTE  = [400, 1350]   # [Fsb1, Fsb2] banda de rejeição (Hz)
JANELA   = 'hamming'
F_RUIDO  = 874.98        # Frequência do ruído dominante (identificada)

# Caminhos de entrada
AUDIO_G1_IN     = 'audio-teste-ruido-G1.wav'         # Áudio do projeto
RUIDO_BRANCO_IN = 'ruido_branco.wav'                  # Ruído gerado no OcenAudio

# Pasta de saída
os.makedirs('graficos', exist_ok=True)

print(f"Fs alvo (kit)     : {FS_KIT} Hz")
print(f"Filtro            : bandstop {F_CORTE[0]}–{F_CORTE[1]} Hz, janela {JANELA}, N={NUMTAPS}")
print(f"Ruído identificado: {F_RUIDO} Hz")

## 1. Projeto do filtro FIR

Uso `scipy.signal.firwin` com os mesmos parâmetros validados no pyfda.
O resultado é **matematicamente equivalente** ao exportado pelo pyfda quando se escolhe
método de janelamento com janela Hamming.

### Justificativa dos parâmetros

Pelo método de janelamento, a **largura mínima de transição** para uma janela Hamming é:

$$ \Delta f_{min} = \frac{K \cdot f_s}{N} = \frac{3{,}3 \times 48000}{199} \approx 795 \text{ Hz} $$

Como o kit limita N < 200, foi necessário alargar a banda de rejeição para aproximadamente
[400, 1350] Hz (≈ 950 Hz de largura), centrada no ruído em 874,98 Hz, para garantir
atenuação significativa no ponto do ruído.

In [ ]:
taps = signal.firwin(NUMTAPS, F_CORTE,
                     pass_zero='bandstop',
                     window=JANELA,
                     fs=FS_KIT)

print(f"Número de coeficientes : {len(taps)}")
print(f"Soma dos coeficientes  : {taps.sum():.6f}   (≈1 fora da banda de rejeição)")
print(f"Simetria (fase linear) : {np.allclose(taps, taps[::-1])}")
print(f"Max |h[n]|             : {np.max(np.abs(taps)):.6f}")

### 1.1 Resposta em frequência do filtro

In [ ]:
w, h = signal.freqz(taps, [1], worN=8192, fs=FS_KIT)
mag_dB = 20 * np.log10(np.abs(h) + 1e-12)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7))
fig.suptitle('Resposta em frequência — FIR rejeita-faixa',
             fontsize=12, fontweight='bold')

for ax, xlim, titulo in [(ax1, (0, 3000), 'Zoom na banda de rejeição (0–3 kHz)'),
                          (ax2, (0, FS_KIT/2), f'Resposta completa (0 até Nyquist = {FS_KIT//2} Hz)')]:
    ax.plot(w, mag_dB, 'b', linewidth=1.2)
    ax.axvline(F_CORTE[0], color='k', ls='--', alpha=0.6, label=f'Fsb1 = {F_CORTE[0]} Hz')
    ax.axvline(F_CORTE[1], color='k', ls='--', alpha=0.6, label=f'Fsb2 = {F_CORTE[1]} Hz')
    ax.axvline(F_RUIDO, color='r', ls=':', alpha=0.8, label=f'Ruído = {F_RUIDO} Hz')
    ax.set_xlim(xlim)
    ax.set_ylim(-100, 5)
    ax.set_xlabel('Frequência (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(titulo)
    ax.grid(True, alpha=0.3)
    if ax is ax1:
        ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('graficos/01_resposta_em_frequencia_FIR.png', dpi=110, bbox_inches='tight')
plt.show()

# Atenuação exatamente na frequência do ruído
idx = np.argmin(np.abs(w - F_RUIDO))
print(f'\nAtenuação em {F_RUIDO} Hz: {mag_dB[idx]:.2f} dB')

## 2. Geração do `coeffs_FIR.csv` e `coeffs_FIR.h`

- O CSV é exportado **vertical** (uma coluna, sem cabeçalho) — mesmo formato
  que o pyfda produz quando `Table Orientation = Vertical`.
- O `.h` é gerado pela função `fir_header()` (retirada do script
  `Le_CSV_Pyfda_gera_coeff_FIR.py` do professor) e contém:
  - `firCoeffs32[]` em `float32_t` (ponto flutuante)
  - `firCoeffsQ15[]` em `q15_t` (ponto fixo 1.15)
  - `NUM_TAPS` (padded para ficar par, exigência do CMSIS-DSP)
  - `filterType = FIR_FLOAT32`

In [ ]:
# Exporta CSV vertical (compatível com pyfda)
pd.DataFrame(taps).to_csv('coeffs_FIR.csv', header=False, index=False)
print('coeffs_FIR.csv gerado.')


def fir_header(fname_out, val):
    """Gera o header CMSIS-DSP (mesma lógica de Le_CSV_Pyfda_gera_coeff_FIR.py)."""
    Ns = len(val)
    if (Ns % 2) != 0:                    # CMSIS-DSP q15 exige NUM_TAPS par
        val = np.append(val, 0)
        Ns  = len(val)
    val_fixo = np.round(val * 2**15)     # converte para Q1.15

    with open(fname_out, 'wt') as f:
        f.write('//define a FIR CMSIS-DSP coefficient array\n\n')
        f.write('#ifndef INCLUDE_COEFFS_FIR_H_\n')
        f.write('#define INCLUDE_COEFFS_FIR_H_\n\n')
        f.write('#include <stdint.h>\n\n')
        f.write('#include "filter.h"\n\n')
        f.write('#ifndef NUM_TAPS\n')
        f.write(f'#define NUM_TAPS {Ns}\n')
        f.write('#endif\n\n')
        f.write('/*************************************/\n')
        f.write('/*     FIR Filter Coefficients       */\n')
        f.write(f'const float32_t firCoeffs32[{Ns}] = {{ ')
        for k in range(Ns):
            sep = ', ' if k < Ns - 1 else ' '
            f.write(f' {val[k]:+-13e}{sep}')
        f.write('};\n\n')
        f.write(f'const q15_t firCoeffsQ15[{Ns}] = {{ ')
        for k in range(Ns):
            sep = ', ' if k < Ns - 1 else ' '
            f.write(f' {int(val_fixo[k])}{sep}')
        f.write('};\n')
        f.write('FilterTypeDef filterType=FIR_FLOAT32;\n')
        f.write('/***********************************/\n\n')
        f.write('#endif /* INCLUDE_COEFFS_FIR_H_ */')


# Lê o CSV e gera o .h (mesmo fluxo do script original)
valores = pd.read_csv('coeffs_FIR.csv', header=None).values[:, 0]
fir_header('coeffs_FIR.h', valores)
print(f'coeffs_FIR.h gerado — NUM_TAPS final: {len(valores) + (1 if len(valores) % 2 else 0)}')

## 3. Funções auxiliares para validação

In [ ]:
def aplicar_fir(sinal, coefs):
    """Aplica o FIR e compensa o atraso de grupo (N-1)/2 para alinhar saída ↔ entrada."""
    atraso = (len(coefs) - 1) // 2
    saida  = signal.lfilter(coefs, 1.0, sinal)
    saida_alinhada = np.concatenate((saida[atraso:], np.zeros(atraso)))
    return saida_alinhada


def carregar_e_reamostrar(path_in, fs_alvo=FS_KIT):
    """Carrega um .wav e reamostra para fs_alvo (48 kHz) se necessário."""
    x_orig, fs_orig = librosa.load(path_in, sr=None, mono=True)
    print(f'  Entrada : {os.path.basename(path_in)}  Fs={fs_orig} Hz  {len(x_orig)/fs_orig:.2f} s')
    if fs_orig != fs_alvo:
        x = librosa.resample(x_orig, orig_sr=fs_orig, target_sr=fs_alvo)
        print(f'  Reamostrado para {fs_alvo} Hz ({len(x)} amostras)')
    else:
        x = x_orig
    return x.astype(np.float32)


def plot_fft_antes_depois(x_in, x_out, fs, titulo, fname, xlim_hz=3000):
    N_in, N_out = len(x_in), len(x_out)
    f_in  = fftfreq(N_in,  1/fs)[:N_in//2]
    f_out = fftfreq(N_out, 1/fs)[:N_out//2]
    X_in  = 20*np.log10(np.abs(fft(x_in)[:N_in//2])  * 2/N_in  + 1e-12)
    X_out = 20*np.log10(np.abs(fft(x_out)[:N_out//2]) * 2/N_out + 1e-12)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(titulo, fontsize=12, fontweight='bold')
    for ax, f, X, cor, subt in [(ax1, f_in,  X_in,  'steelblue', 'ANTES do filtro'),
                                  (ax2, f_out, X_out, 'crimson',   'DEPOIS do filtro')]:
        ax.plot(f, X, cor, linewidth=0.7)
        ax.axvline(F_RUIDO, color='r', ls=':', alpha=0.7, label=f'{F_RUIDO} Hz')
        ax.set_xlim(0, xlim_hz)
        ax.set_xlabel('Frequência (Hz)')
        ax.set_ylabel('Magnitude (dB)')
        ax.set_title(subt)
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=110, bbox_inches='tight')
    plt.show()


def plot_spec_antes_depois(x_in, x_out, fs, titulo, fname, ylim_hz=3000):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(titulo, fontsize=12, fontweight='bold')
    for ax, x, subt in [(ax1, x_in, 'ANTES do filtro'), (ax2, x_out, 'DEPOIS do filtro')]:
        ax.specgram(x, NFFT=2048, Fs=fs, noverlap=1024, cmap='viridis')
        ax.set_ylim(0, ylim_hz)
        ax.set_xlabel('Tempo (s)')
        ax.set_ylabel('Frequência (Hz)')
        ax.set_title(subt)
        ax.axhline(F_RUIDO, color='r', ls=':', alpha=0.8, linewidth=1)
    plt.tight_layout()
    plt.savefig(fname, dpi=110, bbox_inches='tight')
    plt.show()

## 4. Validação 1 — Áudio `audio-teste-ruido-G1.wav`

Espera-se observar:
- No tempo, a amplitude média do sinal **reduz** (ruído contínuo foi removido)
- Na FFT, o pico em 874,98 Hz **desaparece**
- No espectrograma, a faixa horizontal do ruído tonal **some**
- Componentes da voz próximas (~500 Hz e >1500 Hz) são preservadas;
  a região entre 400–1350 Hz sofre alargamento inevitável (limite de N=199)

In [ ]:
print('Processando áudio G1')
print('=' * 55)
x_g1  = carregar_e_reamostrar(AUDIO_G1_IN)
y_g1  = aplicar_fir(x_g1, taps)

sf.write('audio_G1_original_48k.wav', x_g1, FS_KIT, subtype='PCM_16')
sf.write('audio_G1_filtrado_48k.wav', y_g1, FS_KIT, subtype='PCM_16')
print(f'Atraso do FIR: {(len(taps)-1)//2} amostras ({(len(taps)-1)/2/FS_KIT*1000:.2f} ms)')
print('Salvos: audio_G1_original_48k.wav, audio_G1_filtrado_48k.wav')

In [ ]:
plot_fft_antes_depois(x_g1, y_g1, FS_KIT,
                      'FFT antes/depois — Áudio G1',
                      'graficos/fft_audio_G1.png')

In [ ]:
plot_spec_antes_depois(x_g1, y_g1, FS_KIT,
                       'Espectrograma antes/depois — Áudio G1',
                       'graficos/spec_audio_G1.png')

## 5. Validação 2 — Ruído branco 200 Hz – 20 kHz

Arquivo gerado no OcenAudio na faixa 200 Hz – 20 kHz.  
Espera-se que a FFT da saída mostre um **"buraco" na banda 400–1350 Hz**
exatamente conforme a resposta em frequência do filtro — é o teste clássico
de *sweep* (aqui usando ruído branco limitado em banda, que carrega potência em todas as frequências).

In [ ]:
print('Processando ruído branco')
print('=' * 55)

if not os.path.exists(RUIDO_BRANCO_IN):
    print(f'[AVISO] arquivo não encontrado: {RUIDO_BRANCO_IN}')
    print('Anexe o arquivo gerado no OcenAudio e rode novamente esta seção.')
else:
    x_rb = carregar_e_reamostrar(RUIDO_BRANCO_IN)
    y_rb = aplicar_fir(x_rb, taps)

    sf.write('ruido_branco_original_48k.wav', x_rb, FS_KIT, subtype='PCM_16')
    sf.write('ruido_branco_filtrado_48k.wav', y_rb, FS_KIT, subtype='PCM_16')
    print('Salvos: ruido_branco_original_48k.wav, ruido_branco_filtrado_48k.wav')

    # FFT — zoom na banda de rejeição
    plot_fft_antes_depois(x_rb, y_rb, FS_KIT,
                          'FFT antes/depois — Ruído branco 200 Hz–20 kHz (zoom 0–3 kHz)',
                          'graficos/fft_ruido_branco_zoom.png', xlim_hz=3000)
    # FFT — banda completa
    plot_fft_antes_depois(x_rb, y_rb, FS_KIT,
                          'FFT antes/depois — Ruído branco 200 Hz–20 kHz (banda completa)',
                          'graficos/fft_ruido_branco_full.png', xlim_hz=FS_KIT//2)
    # Espectrograma
    plot_spec_antes_depois(x_rb, y_rb, FS_KIT,
                           'Espectrograma antes/depois — Ruído branco',
                           'graficos/spec_ruido_branco.png', ylim_hz=3000)

## 6. Conclusão e checklist para o kit

### Arquivos gerados
| Arquivo                           | Uso                                               |
|-----------------------------------|---------------------------------------------------|
| `coeffs_FIR.csv`                  | Coeficientes em formato pyfda (referência)        |
| `coeffs_FIR.h`                    | Header CMSIS-DSP — **copiar para o projeto STM32** |
| `audio_G1_original_48k.wav`       | Áudio G1 reamostrado — tocar no PC → LINE-IN do kit |
| `audio_G1_filtrado_48k.wav`       | Referência do resultado esperado                  |
| `ruido_branco_filtrado_48k.wav`   | Referência do resultado esperado p/ ruído branco  |
| `graficos/*.png`                  | FFTs e espectrogramas antes/depois                |

### Passos no kit STM32
1. Substituir o `coeffs_FIR.h` do projeto pelo gerado aqui
2. Compilar e gravar no STM32F4 Discovery
3. Conectar a saída do PC na **LINE-IN** do Wolfson Pi Audio
4. Tocar `audio_G1_original_48k.wav` no PC e gravar a saída do codec
5. Repetir com `ruido_branco_200_20k.wav`
6. Comparar a saída capturada com o resultado simulado neste notebook

### Observações
- **Atenuação na frequência alvo:** ≈ -48 dB em 874,98 Hz → o ruído tonal fica
  ~250 vezes mais fraco em amplitude, imperceptível na escuta
- **Fase linear:** o FIR projetado é simétrico, preservando forma de onda (só desloca no tempo)
- **Atraso de grupo:** (N-1)/2 = 99 amostras @ 48 kHz = **2,06 ms** — imperceptível
- **Limitação conhecida:** a banda de rejeição de 950 Hz é larga por causa do
  limite N<200 do kit (ver conclusão detalhada no notebook de análise)